# NeuralForecast Model Comparison

This notebook reads experiment settings from `configs/experiment.yaml` in this repository and compares `GRU`, `TimeXer`, and `iTransformer`.

[Open repository](https://github.com/Jaeho777/newoil)

Outputs:
- train / validation loss curves
- forecast plots
- metrics tables (`MAE`, `RMSE`, `MAPE`, `sMAPE`)


In [ ]:
%pip -q install "git+https://github.com/Nixtla/neuralforecast.git" utilsforecast openpyxl pyyaml


In [ ]:
import os
import shutil
import subprocess
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml

from neuralforecast import NeuralForecast
from neuralforecast.models import GRU, TimeXer, iTransformer

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.float_format = "{:,.4f}".format

REPO_URL = "https://github.com/Jaeho777/newoil.git"
WORKDIR = Path("/content/newoil")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORKDIR)], check=True)

CONFIG_PATH = WORKDIR / "configs" / "experiment.yaml"
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Loaded config: {CONFIG_PATH}")
cfg


In [ ]:
data_cfg = cfg["data"]
exp_cfg = cfg["experiment"]
plot_cfg = cfg.get("plot", {})

DATA_FORMAT = data_cfg["format"]
FILE_PATH = data_cfg.get("file_path") or None
SHEET_NAME = data_cfg.get("sheet_name", 0)
FREQ = data_cfg["freq"]
TIME_COL = data_cfg["time_col"]
ID_COL = data_cfg.get("id_col")
TARGET_COL = data_cfg["target_col"]

H = int(exp_cfg["horizon"])
VAL_SIZE = int(exp_cfg["val_size"])
TEST_SIZE = int(exp_cfg["test_size"])
PATCH_LEN = int(exp_cfg.get("patch_len", 16))
LOOKBACK_MULTIPLIER = int(plot_cfg.get("lookback_multiplier", 2))


def default_input_size(h, patch_len=16):
    base = max(3 * h, patch_len)
    return int(np.ceil(base / patch_len) * patch_len)


INPUT_SIZE = default_input_size(H, PATCH_LEN)
LOOKBACK_PLOT = INPUT_SIZE * LOOKBACK_MULTIPLIER
MODEL_NAMES = ["GRU", "TimeXer", "iTransformer"]

if FILE_PATH is not None:
    FILE_PATH = str((WORKDIR / FILE_PATH).resolve()) if not Path(FILE_PATH).is_absolute() else FILE_PATH

print(f"Computed input_size: {INPUT_SIZE}")
print(f"Plot lookback: {LOOKBACK_PLOT}")
print(f"Configured file path: {FILE_PATH}")


In [ ]:
def load_source_file(file_path=None, sheet_name=0):
    if file_path is None:
        from google.colab import files

        uploaded = files.upload()
        if not uploaded:
            raise ValueError("No file was uploaded.")
        file_path = next(iter(uploaded))
    else:
        file_path = Path(file_path)
        if not file_path.exists():
            raise FileNotFoundError(f"Configured file was not found: {file_path}")

    lower_path = str(file_path).lower()
    if lower_path.endswith(".csv"):
        raw_df = pd.read_csv(file_path)
    elif lower_path.endswith(".xlsx") or lower_path.endswith(".xls"):
        raw_df = pd.read_excel(file_path, sheet_name=sheet_name)
    else:
        raise ValueError("Only .csv, .xls, and .xlsx files are supported.")

    return raw_df, str(file_path)


raw_df, resolved_file = load_source_file(FILE_PATH, SHEET_NAME)
print(f"Loaded file: {resolved_file}")
display(raw_df.head())


In [ ]:
def prepare_neuralforecast_df(raw_df, data_format, time_col, target_col, id_col=None):
    frame = raw_df.copy()

    if data_format == "wide":
        if time_col not in frame.columns:
            raise ValueError(f"Missing time column: {time_col}")
        df = frame.melt(id_vars=[time_col], var_name="unique_id", value_name="y")
        df = df.rename(columns={time_col: "ds"})

    elif data_format == "long":
        required = {time_col, target_col}
        missing = required - set(frame.columns)
        if missing:
            raise ValueError(f"Missing required columns: {sorted(missing)}")

        working_id_col = id_col
        if working_id_col is None or working_id_col not in frame.columns:
            working_id_col = "__series_id"
            frame[working_id_col] = "series_1"

        df = frame.rename(
            columns={
                time_col: "ds",
                working_id_col: "unique_id",
                target_col: "y",
            }
        )[["unique_id", "ds", "y"]]

    else:
        raise ValueError("data.format must be 'long' or 'wide'.")

    df = df.dropna(subset=["y"]).copy()
    df["ds"] = pd.to_datetime(df["ds"])

    if df.duplicated(subset=["unique_id", "ds"]).any():
        raise ValueError("Found duplicated (unique_id, ds) rows.")

    panel = df.pivot(index="ds", columns="unique_id", values="y").sort_index()
    panel = panel.dropna(how="any")
    if panel.empty:
        raise ValueError("No aligned observations remain after dropping timestamps with missing values.")

    prepared = (
        panel.reset_index()
        .melt(id_vars="ds", var_name="unique_id", value_name="y")
        .sort_values(["unique_id", "ds"])
        .reset_index(drop=True)
    )
    return prepared


df = prepare_neuralforecast_df(
    raw_df=raw_df,
    data_format=DATA_FORMAT,
    time_col=TIME_COL,
    target_col=TARGET_COL,
    id_col=ID_COL,
)

display(df.head())
display(df.groupby("unique_id").agg(points=("y", "size")))


In [ ]:
series_lengths = df.groupby("unique_id").size()
if series_lengths.nunique() != 1:
    raise ValueError("All series must have the same length after alignment.")

series_size = int(series_lengths.iloc[0])
n_series = df["unique_id"].nunique()
train_points = series_size - VAL_SIZE - TEST_SIZE

if train_points <= 0:
    raise ValueError("Not enough observations after validation and test splits.")

if train_points < INPUT_SIZE:
    raise ValueError(
        f"Train points per series ({train_points}) are smaller than input_size ({INPUT_SIZE}). "
        "Use a longer history or reduce horizon / splits."
    )

train_val_df = (
    df.groupby("unique_id", group_keys=False)
    .head(series_size - TEST_SIZE)
    .reset_index(drop=True)
)

test_df = (
    df.groupby("unique_id", group_keys=False)
    .tail(TEST_SIZE)
    .reset_index(drop=True)
)

print(f"Number of series: {n_series}")
print(f"Observations per series after alignment: {series_size}")
print(f"Training points per series: {train_points}")
print(f"Validation points per series: {VAL_SIZE}")
print(f"Test points per series: {TEST_SIZE}")

display(train_val_df.head())
display(test_df.head())


In [ ]:
models = [
    GRU(h=H, input_size=INPUT_SIZE),
    TimeXer(h=H, input_size=INPUT_SIZE, n_series=n_series),
    iTransformer(h=H, input_size=INPUT_SIZE, n_series=n_series),
]

nf = NeuralForecast(models=models, freq=FREQ)
nf.fit(df=train_val_df, val_size=VAL_SIZE)

fitted_models = {model.__class__.__name__: model for model in nf.models}
list(fitted_models.keys())


In [ ]:
def plot_loss_curves(model):
    train_df = pd.DataFrame(model.train_trajectories, columns=["step", "train_loss"])
    valid_df = pd.DataFrame(model.valid_trajectories, columns=["step", "valid_loss"])

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(train_df["step"], train_df["train_loss"], label="Train loss", linewidth=1.5)

    if not valid_df.empty:
        ax.plot(valid_df["step"], valid_df["valid_loss"], label="Validation loss", linewidth=2, marker="o")

    ax.set_title(f"{model.__class__.__name__} loss curves")
    ax.set_xlabel("Global step")
    ax.set_ylabel("Loss")
    ax.legend()
    plt.show()

    return train_df, valid_df


loss_frames = {}
for model_name in MODEL_NAMES:
    train_df, valid_df = plot_loss_curves(fitted_models[model_name])
    loss_frames[model_name] = {"train": train_df, "valid": valid_df}

loss_frames["GRU"]["train"].head()


In [ ]:
preds_df = nf.predict()
preds_df = preds_df.sort_values(["unique_id", "ds"]).reset_index(drop=True)
eval_df = test_df.merge(preds_df, on=["unique_id", "ds"], how="left")
display(eval_df.head())


def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mape(y_true, y_pred):
    return float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)


def smape(y_true, y_pred):
    denom = np.abs(y_true) + np.abs(y_pred)
    denom = np.where(denom == 0, 1e-8, denom)
    return float(np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100)


overall_rows = []
per_series_rows = []

for model_name in MODEL_NAMES:
    y_true = eval_df["y"].to_numpy()
    y_pred = eval_df[model_name].to_numpy()

    overall_rows.append(
        {
            "model": model_name,
            "MAE": mae(y_true, y_pred),
            "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred),
            "sMAPE": smape(y_true, y_pred),
        }
    )

    for unique_id, group in eval_df.groupby("unique_id"):
        y_true_group = group["y"].to_numpy()
        y_pred_group = group[model_name].to_numpy()
        per_series_rows.append(
            {
                "unique_id": unique_id,
                "model": model_name,
                "MAE": mae(y_true_group, y_pred_group),
                "RMSE": rmse(y_true_group, y_pred_group),
                "MAPE": mape(y_true_group, y_pred_group),
                "sMAPE": smape(y_true_group, y_pred_group),
            }
        )

overall_metrics_df = pd.DataFrame(overall_rows).set_index("model").sort_values("RMSE")
per_series_metrics_df = pd.DataFrame(per_series_rows).sort_values(["unique_id", "RMSE"])

display(overall_metrics_df)
display(per_series_metrics_df)


In [ ]:
def plot_predictions(model_name, history_df, actual_df, pred_df, lookback):
    unique_ids = actual_df["unique_id"].unique()
    fig, axes = plt.subplots(len(unique_ids), 1, figsize=(12, 4 * len(unique_ids)))

    if len(unique_ids) == 1:
        axes = [axes]

    for ax, unique_id in zip(axes, unique_ids):
        history_slice = history_df[history_df["unique_id"] == unique_id].tail(lookback)
        actual_slice = actual_df[actual_df["unique_id"] == unique_id]
        pred_slice = pred_df[pred_df["unique_id"] == unique_id]

        ax.plot(history_slice["ds"], history_slice["y"], label="History", color="black", linewidth=1.5)
        ax.plot(actual_slice["ds"], actual_slice["y"], label="Actual", color="#1f77b4", linewidth=2)
        ax.plot(
            pred_slice["ds"],
            pred_slice[model_name],
            label="Forecast",
            color="#d62728",
            linestyle="--",
            linewidth=2,
            marker="o",
        )
        ax.axvline(actual_slice["ds"].min(), color="gray", linestyle=":")
        ax.set_title(f"{model_name} forecast - {unique_id}")
        ax.set_xlabel("Date")
        ax.set_ylabel("Target")
        ax.legend()

    plt.tight_layout()
    plt.show()


for model_name in MODEL_NAMES:
    plot_predictions(model_name, train_val_df, test_df, eval_df, LOOKBACK_PLOT)
